In [1]:
%pip install -q faiss-gpu

%pip install -q -U sentence_transformers rank_bm25 PyStemmer ranknexus

!nvidia-smi

Mon Nov  2 07:13:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             49W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset

dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")

dataset["train"]

README.md:   0%|          | 0.00/525 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.30MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  110kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4738 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/539 [00:00<?, ? examples/s]

Dataset({
    features: ['doc_id', 'link', 'title', 'text', 'query'],
    num_rows: 4738
})

In [3]:
from sentence_transformers import SentenceTransformer
from datasets import Dataset

retriever = SentenceTransformer("pameydorke/arcanumchat-retriever-bge-m3-v2")

train_split = dataset["train"]
val_split = dataset["validation"]

corpus_texts = list(dict.fromkeys(train_split["text"])) + list(dict.fromkeys(val_split["text"]))
text_to_idx = {t: i for i, t in enumerate(corpus_texts)}

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/42.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (3 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

In [4]:
import Stemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_stemmer = Stemmer.Stemmer("english")

def tokenize(text: str):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    return _stemmer.stemWords(tokens)

tokenized_corpus = [tokenize(doc) for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

In [5]:
corpus_embeddings = retriever.encode(
    corpus_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [6]:
from ranknexus import fuse
from sentence_transformers import util
import torch
import numpy as np
from tqdm import tqdm

def mine_hybrid_hard_negatives(
    dataset,
    retriever,
    corpus_texts,
    corpus_embeddings,
    text_to_idx,
    num_negatives=4,
    k_dense=20,
    k_bm25=20,
    rrf_k=60,
    hard_cutoff=10,
    medium_cutoff=30,
    hard_frac=0.5,
    medium_frac=0.3,
    query_batch_size=128,
    sim_batch_size=256,
    seed=42,
):
    """Mine reranker training negatives from a dense ∪ BM25 RRF pool.

    Returns a dataset with columns (anchor, positive, negative) — one row
    per (query, positive, negative) triplet, matching the `labeled-pair`
    format that CrossEncoder training expects.
    """
    queries   = dataset["anchor"]
    positives = dataset["positive"]

    query_embeddings = retriever.encode(
        queries,
        batch_size=query_batch_size,
        show_progress_bar=True,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    # --- Dense top-k in batches (avoids an [n_queries, n_corpus] blowup) ---
    dense_topk = []
    for start in range(0, len(query_embeddings), sim_batch_size):
        batch = query_embeddings[start:start + sim_batch_size]
        sim = util.cos_sim(batch, corpus_embeddings)
        topk = torch.argsort(sim, dim=1, descending=True)[:, :k_dense]
        dense_topk.append(topk.cpu().numpy())
    dense_topk = np.vstack(dense_topk)                 # [n_queries, k_dense]

    # --- Per-query: dense ∪ BM25 → RRF → sample negatives ---
    rows = []
    for i, (query, positive) in enumerate(tqdm(
        zip(queries, positives), total=len(queries), desc="Mining hybrid"
    )):
        positive_idx = text_to_idx.get(positive)

        dense_ranking = [{"id": int(idx)} for idx in dense_topk[i]]

        scores = bm25.get_scores(tokenize(query))
        bm25_topk = np.argsort(scores)[::-1][:k_bm25]
        bm25_ranking = [{"id": int(idx)} for idx in bm25_topk]

        fused = fuse(
            [dense_ranking, bm25_ranking],
            method="rrf",
            k=rrf_k,
            top_k=None,
        )
        fused_ids = [r.document_id for r in fused if r.document_id != positive_idx]

        # Difficulty bins from the fused pool
        hard_pool   = fused_ids[:hard_cutoff]
        medium_pool = fused_ids[hard_cutoff:medium_cutoff]
        easy_pool   = fused_ids[medium_cutoff:]

        n_hard   = max(0, round(num_negatives * hard_frac))
        n_medium = max(0, round(num_negatives * medium_frac))
        n_easy   = num_negatives - n_hard - n_medium

        rng = np.random.default_rng(seed + i)

        def sample(pool, n):
            if not pool or n <= 0:
                return []
            k = min(n, len(pool))
            return rng.choice(pool, size=k, replace=False).tolist()

        neg_ids  = sample(hard_pool,   n_hard)
        neg_ids += sample(medium_pool, n_medium)
        neg_ids += sample(easy_pool,   n_easy)

        # Fallback: random corpus docs if fused pool was too small
        while len(neg_ids) < num_negatives:
            j = int(rng.integers(0, len(corpus_texts)))
            if j != positive_idx and j not in neg_ids:
                neg_ids.append(j)

        rows.append({"anchor": query, "document": positive, "label": 1})
        
        for idx in neg_ids[:num_negatives]:
            rows.append({"anchor": query, "document": corpus_texts[int(idx)], "label": 0})

    return Dataset.from_list(rows)

In [7]:
train_dataset = dataset["train"].select_columns(["query", "text"]).rename_columns({
    "query": "anchor", "text": "positive",
})

hard_train_dataset = mine_hybrid_hard_negatives(
    dataset=train_dataset,
    retriever=retriever,
    corpus_texts=corpus_texts,
    corpus_embeddings=corpus_embeddings,
    text_to_idx=text_to_idx,
    num_negatives=4,
)

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Mining hybrid: 100%|██████████| 4738/4738 [00:07<00:00, 646.72it/s]


In [16]:
hard_train_dataset

Dataset({
    features: ['anchor', 'document', 'label'],
    num_rows: 23690
})

In [8]:
val_dataset = dataset["validation"].select_columns(["query", "text"]).rename_columns({
    "query": "anchor", "text": "positive",
})

hard_val_dataset = mine_hybrid_hard_negatives(
    dataset=val_dataset,
    retriever=retriever,
    corpus_texts=corpus_texts,
    corpus_embeddings=corpus_embeddings,
    text_to_idx=text_to_idx,
    num_negatives=4,
)

hard_val_dataset

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Mining hybrid: 100%|██████████| 539/539 [00:00<00:00, 634.53it/s]


Dataset({
    features: ['anchor', 'document', 'label'],
    num_rows: 2695
})

In [9]:
from collections import defaultdict

grouped = defaultdict(lambda: {"positive": [], "negative": []})

for anchor, document, label in zip(
    hard_val_dataset["anchor"], hard_val_dataset["document"], hard_val_dataset["label"]
):
    key = "positive" if int(label) == 1 else "negative"
    grouped[anchor][key].append(document)

# Only keep queries that have at least one positive AND one negative
hard_eval_samples = [
    {"query": q, **docs}
    for q, docs in grouped.items()
    if docs["positive"] and docs["negative"]
]

print(f"Evaluation queries: {len(hard_eval_samples)}")

Evaluation queries: 539


In [10]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

evaluator = CrossEncoderRerankingEvaluator(
    samples=hard_eval_samples,
    at_k=20,
    name="val-hard-negatives",
    batch_size=64,
    show_progress_bar=True,
    always_rerank_positives=True,
)

In [11]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [12]:
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from datasets import Dataset

train_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_train_dataset],
    "document": [p["document"] for p in hard_train_dataset],
    "label": [p["label"] for p in hard_train_dataset]
})

val_dataset_formatted = Dataset.from_dict({
    "query": [p["anchor"] for p in hard_val_dataset],
    "document": [p["document"] for p in hard_val_dataset],
    "label": [p["label"] for p in hard_val_dataset]
})

train_loss = BinaryCrossEntropyLoss(model)

In [14]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from transformers import EarlyStoppingCallback

args = CrossEncoderTrainingArguments(
    warmup_steps=0.1,
    lr_scheduler_type="linear",

    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps = 2,

    logging_steps=100,

    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_val-hard-negatives_ndcg@20",
    greater_is_better=True,

    eval_on_start=True,
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,

    evaluator=evaluator,
    loss=train_loss,
)

In [15]:
trainer.train()

Step,Training Loss,Validation Loss,Val-hard-negatives Map,Val-hard-negatives Mrr@20,Val-hard-negatives Ndcg@20
0,No log,0.364489,0.843816,0.887415,0.917853
100,0.305102,0.253932,0.857669,0.902505,0.928591
200,0.307232,0.256608,0.858442,0.904515,0.929072
300,0.292254,0.256809,0.846846,0.891064,0.919645
400,0.285113,0.248558,0.855968,0.902041,0.927543
500,0.287406,0.240499,0.860451,0.904669,0.930468
600,0.280357,0.234069,0.854886,0.900031,0.925994
700,0.248495,0.236017,0.855257,0.900711,0.927077
741,0.248495,0.232744,0.857205,0.902659,0.928528


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (10 outputs left out)

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

TrainOutput(global_step=741, training_loss=0.28489748894283484, metrics={'train_runtime': 3773.9882, 'train_samples_per_second': 6.277, 'train_steps_per_second': 0.196, 'total_flos': 0.0, 'train_loss': 0.28489748894283484, 'epoch': 1.0})

In [17]:
model.push_to_hub("pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-hybrid-search")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'https://huggingface.co/pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-hybrid-search/commit/ed7f0bb69fd8e05a2f69dd62de94569ab9d08f0d'